# State-Space Modeling

**State-space representation** keeps track of every internal variable
explicitly, through a set of first-order differential equations:

$$
\dot{\mathbf{x}} = A\mathbf{x} + B u, \qquad y = C\mathbf{x} + Du
$$

where $\mathbf{x}$ is the *state vector*. This page builds the state-space
model symbolically with SymPy, two different ways, and then uses it for
controllability, observability, and pole-placement design.

In [2]:
from IPython.core.display import HTML
from sympy import *
from mathprint import *

## 1. Transfer Function to State-Space

Given a strictly-proper transfer function

$$
G(s) = \frac{b_2 s^2 + b_1 s + b_0}{s^3 + a_2 s^2 + a_1 s + a_0}
$$

the **controllable canonical form** can be read directly off the
coefficients. Let's build $A$, $B$, $C$, $D$ symbolically for a general
third-order system, then substitute numbers at the end.


In [3]:
a0, a1, a2 = symbols('a0 a1 a2')
b0, b1, b2 = symbols('b0 b1 b2')

n = 3  # system order

A = zeros(n, n)
A[:-1, 1:] = eye(n - 1)
A[-1, :] = Matrix([[-a0, -a1, -a2]])

B = zeros(n, 1)
B[-1, 0] = 1

C = Matrix([[b0, b1, b2]])
D = Matrix([[0]])

mprint("A =", latex(A), ", B =", latex(B))
mprint("C =", latex(C), ", D =", latex(D))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Sanity check: reconstruct $\boxed{G(s) = C(sI - A)^{-1}B + D}$ and confirm it
matches the transfer function we started from.


In [4]:
s = symbols('s')
I3 = eye(n)
G_reconstructed = simplify((C * (s * I3 - A).inv() * B + D)[0])
mprint("G(s)=", latex(G_reconstructed))


<IPython.core.display.Math object>

Now, let us put the whole process of converting a transfer function into a state space in a reusable function:

In [5]:
def tf_to_ss(tf):
    num, denom = tf.as_numer_denom()
    coeffs = denom.as_poly(s).all_coeffs()[::-1]
    n = len(coeffs) - 1 # system order

    coeffs = [-1 * coeffs[a] for a in range(n)] # negate the coefficients for the state-space representation

    A = zeros(n, n)
    A[:-1, 1:] = eye(n - 1)
    A[-1, :] = Matrix([coeffs])

    B = zeros(n, 1)
    B[-1, 0] = 1

    coeffs = num.as_poly(s).all_coeffs()[::-1]
    C = Matrix([coeffs])
    D = Matrix([[0]])
    
    return A, B, C, D

Let's test the `tf_to_ss(...)` function:

In [6]:
A, B, C, D = tf_to_ss(G_reconstructed)

mprint("A =", latex(A), ", B =", latex(B))
mprint("C =", latex(C), ", D =", latex(D))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

### Worked example

Take $G(s) = \dfrac{3s + 5}{s^3 + 6s^2 + 11s + 6}$ (poles at $-1, -2, -3$).


In [ ]:
G  = (3*s+5) / (s**3 + 6*s**2 + 11*s + 6)

A, B, C, D = tf_to_ss(G)
mprint("A =", latex(A), ", B =", latex(B))
mprint("C =", latex(C), ", D =", latex(D))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

## 2. ODEs to State-Space (Physical Derivation)

Canonical form is convenient but abstract — the states don't correspond to
anything physical. It's often more natural to derive $A$, $B$, $C$, $D$
directly from the governing ODEs, the way we did in *System Modeling:
Rectilinear System*.

Recall the mass-spring-damper system:

$$
m\ddot{x} + c\dot{x} + kx = F(t)
$$

Choose states $x_1 = x$ (position) and $x_2 = \dot{x}$ (velocity). Then:

$$
\dot{x}_1 = x_2, \qquad \dot{x}_2 = \frac{1}{m}\left(F - c x_2 - k x_1\right)
$$


In [ ]:
m, c, k, F = symbols('m c k F')
x1, x2 = symbols('x1 x2')

x1_dot = x2
x2_dot = (F - c * x2 - k * x1) / m

A_phys = Matrix([[diff(x1_dot, x1), diff(x1_dot, x2)],
                 [diff(x2_dot, x1), diff(x2_dot, x2)]])

B_phys = Matrix([[diff(x1_dot, F)],
                 [diff(x2_dot, F)]])

mprint("A =", latex(A_phys), ", B =", latex(B_phys))


<IPython.core.display.Math object>

Output is position, so $C = \begin{bmatrix}1 & 0\end{bmatrix}$,
$D = 0$. Note $A_{\text{phys}}$ has exactly the same structure as the
canonical form above once we set $a_0 = k/m$, $a_1 = c/m$ — same physics,
different (equally valid) coordinate choice.


In [ ]:
C_phys = Matrix([[1, 0]])
D_phys = Matrix([[0]])
mprint("C =", latex(C_phys), ", D =", latex(D_phys))


<IPython.core.display.Math object>

## 3. Solving the State Equation Symbolically

The free (zero-input) response is $\mathbf{x}(t) = e^{At}\mathbf{x}(0)$.
SymPy computes the matrix exponential directly for small, numeric $A$.


In [ ]:
t = symbols('t', positive=True)

A_num = Matrix([[0, 1], [-2, -3]])   # poles at -1, -2
x0 = Matrix([[1], [0]])

x_t = simplify((A_num * t).exp() * x0)
x_t


Matrix([
[(2*exp(t) - 1)*exp(-2*t)],
[2*(1 - exp(t))*exp(-2*t)]])

Compare this against the step/impulse response obtained from the
equivalent transfer function on the *Second-Order System* page — both
routes must agree, since they describe the same physical system in
different coordinates.


## 4. Controllability

A system is controllable if the controllability matrix

$$
\mathcal{C} = \begin{bmatrix} B & AB & A^2B & \cdots & A^{n-1}B \end{bmatrix}
$$

has full rank $n$. Let's write a small symbolic helper and test it on a
controllable system, then on a deliberately uncontrollable one.


In [ ]:
def controllability_matrix(A, B):
    n = A.shape[0]
    cols = [B]
    for i in range(1, n):
        cols.append(A**i * B)
    return Matrix.hstack(*cols)

Worked example:

In [ ]:
# Controllable example
A_c = Matrix([[0, 1], [-2, -3]])
B_c = Matrix([[0], [1]])

mprint("A_c =", latex(A_c))
mprint("B_c =", latex(B_c))

print("\n--- Controllability Matrix ---")
Cc = controllability_matrix(A_c, B_c)

mprint(r"\mathcal{C} =", latex(Cc))
print("Rank:", Cc.rank())

<IPython.core.display.Math object>

<IPython.core.display.Math object>


--- Controllability Matrix ---


<IPython.core.display.Math object>

Rank: 2


Another worked example for a deliberately uncontrollable system:

In [ ]:
# Deliberately uncontrollable example:
# input only drives x1, and x1, x2 are decoupled
A_uc = Matrix([[-1, 0], [0, -2]])
B_uc = Matrix([[1], [0]])

mprint("A_uc =", latex(A_uc))
mprint("B_uc =", latex(B_uc))

print("\n--- Controllability Matrix ---")
Cuc = controllability_matrix(A_uc, B_uc)

mprint(r"\mathcal{C} =", latex(Cuc))
print("Rank:", Cuc.rank())


<IPython.core.display.Math object>

<IPython.core.display.Math object>


--- Controllability Matrix ---


<IPython.core.display.Math object>

Rank: 1


The rank drops to 1 — $u$ can never influence $x_2$, since $B$ has
no component in that direction and $A$ is diagonal (decoupled). This is
exactly what "uncontrollable" looks like algebraically.


## 5. Observability

The dual notion: a system is observable if

$$
\mathcal{O} = \begin{bmatrix} C \\ CA \\ CA^2 \\ \vdots \\ CA^{n-1} \end{bmatrix}
$$

has full rank $n$.


In [ ]:
def observability_matrix(A, C):
    n = A.shape[0]
    rows = [C]
    for i in range(1, n):
        rows.append(C * A**i)
    return Matrix.vstack(*rows) 


Worked example for a delibirately unobservable system:

In [ ]:
# Deliberately unobservable example: output only sees x1,
# and x2 doesn't affect x1's dynamics
A_uo = Matrix([[-1, 0], [0, -2]])
C_uo = Matrix([[1, 0]])

mprint("A_uo =", latex(A_uo))
mprint("C_uo =", latex(C_uo))

print("\n--- Observability Matrix ---")
Ouo = observability_matrix(A_uo, C_uo)

mprint(r"\mathcal{O} =", latex(Ouo))
print("Rank:", Ouo.rank())


<IPython.core.display.Math object>

<IPython.core.display.Math object>


--- Observability Matrix ---


<IPython.core.display.Math object>

Rank: 1


Same story as before, mirrored: $x_2$ never shows up in $y$, and
since the system is decoupled, its motion is invisible from the output.


## 6. Pole Placement (State Feedback) with  Ackerman

With full-state feedback $u = -K\mathbf{x}$, the closed-loop dynamics
become $\dot{\mathbf{x}} = (A - BK)\mathbf{x}$. We choose $K$ so that
$A - BK$ has a desired characteristic polynomial (desired closed-loop)
poles.

**Ackermann's formula** gives $K$ directly, purely symbolically:

$$
K = \begin{bmatrix}0 & \cdots & 0 & 1\end{bmatrix} \mathcal{C}^{-1} \; \phi_d(A)
$$

where $\phi_d(A)$ is the desired characteristic polynomial evaluated at the
matrix $A$, and $\mathcal{C}$ is the controllability matrix (which must be
invertible $\rightarrow$ the system must be controllable!).


In [ ]:
def ackermann(A, B, desired_poles):
    n = A.shape[0]
    s = symbols('s')

    # Desired characteristic polynomial phi_d(s)
    phi_d = prod([(s - p) for p in desired_poles])
    phi_d = expand(phi_d)
    phi_d_poly = Poly(phi_d, s)

    # phi_d(A) via Cayley-Hamilton-style substitution
    coeffs = phi_d_poly.all_coeffs()  # highest degree first
    phi_d_A = zeros(n, n)
    deg = len(coeffs) - 1
    for i, c in enumerate(coeffs):
        power = deg - i
        phi_d_A += c * A**power

    Cmat = controllability_matrix(A, B)
    e_n = zeros(1, n)
    e_n[0, -1] = 1

    K = e_n * Cmat.inv() * phi_d_A
    return simplify(K)


Worked example:

In [ ]:
# Plant: A_c, B_c from Section 4 (poles currently at -1, -2)
# Controllable example
A_c = Matrix([[0, 1], [-2, -3]])
B_c = Matrix([[0], [1]])

desired_poles = [-4, -6]
K = ackermann(A_c, B_c, desired_poles)
K

Matrix([[22, 7]])

Verify: the eigenvalues of $A - BK$ should now match the desired
poles exactly.


In [ ]:
A_cl = A_c - B_c * K
eigs = simplify(A_cl.eigenvals())
list(eigs.keys())

[-6, -4]